# 💧 LFM3-Omni - Any Input, Text or Speech Output

This notebook walks through what [`LiquidAI/LFM3-Omni-2B`](https://huggingface.co/LiquidAI/LFM3-Omni-2B) can do with 🤗 Transformers.
LFM3-Omni is a single end-to-end model that **understands text, images, video and audio**, and **answers in text, speech,
or interleaved text and speech**. One small model replaces a separate ASR model, vision-language model, LLM and TTS model.

## 🎯 What You'll Find:
- **Speech** - recognition, synthesis, translation and spoken chat
- **Images** - captioning and visual question answering, asked in text or by voice, answered in text or speech
- **Video** - questions about a clip, using the frames alone or the frames and their soundtrack
- **Tool calling** - function calls from text or spoken requests
- **Output control** - let the model choose its output modality, or force text, speech or interleaved output

## 📋 Prerequisites:
- **GPU Runtime**: Select GPU in `Runtime` → `Change runtime type` (an L4 or A100 is recommended, the model runs in bfloat16)
- **Hugging Face Account**: For accessing models and datasets

# 📦 Installation & Setup

First, let's install all the required packages. `torchcodec` decodes video (and the soundtrack of a video).

In [ ]:
!pip install -qqq "transformers>=5.20.2" accelerate datasets torchcodec pillow librosa soundfile --progress-bar off

Let's now verify the packages are installed correctly

In [ ]:
import torch
import transformers

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"🖥️ GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'none'}")

# Loading the model from Transformers 🤗

The processor bundles the tokenizer, the image, video and audio preprocessors, and the **Liquid codec**
(`processor.audio_tokenizer`) that turns generated speech codes into a 24 kHz waveform.

In [ ]:
import torch
from PIL import Image
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Omni-2B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

print("🧠 Loading model...")
model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=dtype, device_map=device, trust_remote_code=True
).eval()

print("📚 Loading processor...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device=device).eval()

print("\n✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

## 🛠️ A small helper

Every example below is a chat conversation. The helper renders it (loading any image, video or audio it names),
generates, and returns the text reply and the speech, if any.

The model **chooses its output modality itself**, guided by the system prompt. Pass `generation_mode="text"`, `"audio"`
or `"interleaved"` to force one.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

SR = 24_000  # generated speech is 24 kHz


@torch.inference_mode()
def generate(conversation, generation_mode=None, max_new_tokens=512, **processor_kwargs):
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs=processor_kwargs,
    ).to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        generation_mode=generation_mode,
        audio_temperature=0.4,
        audio_repetition_penalty=1.2,
        audio_repetition_window=64,
    )
    prompt_len = inputs["input_ids"].shape[1]
    reply = processor.decode(out.sequences[0, prompt_len : out.sequence_lengths[0]], skip_special_tokens=True)
    speech = None
    if out.audio_lengths[0] > 0:
        speech = codec.decode(out.audio_codes[:1, :, : out.audio_lengths[0]]).audio_values[0].float().cpu().numpy().squeeze()
    return reply, speech


def play(waveform, sr=SR):
    display(Audio(waveform, rate=sr))

# 🗣️ Speech

## Text-to-speech

System prompt: `Read this with a {female|male} voice in {language}.` Supported languages: English (American and British),
Arabic, Chinese, Danish, Dutch, Finnish, French, German, Greek, Hebrew, Italian, Japanese, Korean, Polish, Portuguese, Spanish, Swedish and Turkish.

In [ ]:
torch.manual_seed(0)
_, speech = generate([
    {"role": "system", "content": "Read this with a female voice in American English."},
    {"role": "user", "content": "What is this tiny garden visitor doing?"},
])
sf.write("question.wav", speech, SR)
play(speech)

In [ ]:
_, speech_de = generate([
    {"role": "system", "content": "Read this with a male voice in German."},
    {"role": "user", "content": "Heute probieren wir etwas Neues: Wir malen Tiere mit geschlossenen Augen. Mein erster Versuch sieht aus wie eine Kartoffel mit Ohren."},
])
sf.write("german.wav", speech_de, SR)
play(speech_de)

## Speech recognition

System prompt: `Perform ASR in {language}.` Audio goes in the user turn as `{"type": "audio", "audio": <path or URL>}`; it is resampled to 16 kHz mono on load.

In [ ]:
for path, language in [("question.wav", "English"), ("german.wav", "German")]:
    transcript, _ = generate([
        {"role": "system", "content": f"Perform ASR in {language}."},
        {"role": "user", "content": [{"type": "audio", "audio": path}]},
    ])
    print(f"🎙️ [{language}] {transcript}")

## Speech translation

Translation works best into or out of English.

| Direction | System prompt |
|---|---|
| Speech → text | `Convert the speech in {source} to written {target}.` |

In [ ]:
translation, _ = generate([
    {"role": "system", "content": "Convert the speech in German to written English."},
    {"role": "user", "content": [{"type": "audio", "audio": "german.wav"}]},
])
print("🌍 Speech → text:", translation)

## Spoken chat

System prompt: `Respond with interleaved text and audio.` The model answers with text and speech interleaved,
so speech can start playing before the full answer is written. The question can be text or speech.

In [ ]:
reply, speech = generate(
    [
        {"role": "system", "content": "Respond with interleaved text and audio."},
        {"role": "user", "content": "I want to move more, but I have not found an activity I enjoy. How can I try a few options and build a small habit? Give two practical ideas in short, friendly sentences."},
    ],
    max_new_tokens=1024,
)
print("💬", reply)
play(speech)

For plain text chat, use no system prompt (or `Respond with text.`).

In [ ]:
reply, _ = generate([{"role": "user", "content": "Write a haiku about a small model that can see and hear."}])
print("💬", reply)

# 🖼️ Images

Images are processed at their native resolution. Recommended system prompts:
`Describe the provided image.` for captions, and `Answer questions about the provided image. Base every answer only on what can be seen.` for questions.

In [ ]:
from transformers.image_utils import load_image

image = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg")
image.save("bee.jpg")
display(image.resize((image.width // 3, image.height // 3)))

caption, _ = generate([
    {"role": "system", "content": "Describe the provided image."},
    {"role": "user", "content": [{"type": "image", "image": "bee.jpg"}]},
])
print("🖼️", caption)

The question can also be **asked by voice**, with the recording in place of the text. Here we use the spoken question we synthesized earlier.
When the answer should be in a different modality from the question, add an output instruction after the task prompt, e.g. `Respond with text.`

In [ ]:
play(sf.read("question.wav")[0])
answer, _ = generate([
    {"role": "system", "content": "Answer questions about the provided image. Base every answer only on what can be seen. Respond with text."},
    {"role": "user", "content": [{"type": "image", "image": "bee.jpg"}, {"type": "audio", "audio": "question.wav"}]},
])
print("💡", answer)

Or ask in text and get a **spoken** answer:

In [ ]:
answer, speech = generate([
    {"role": "system", "content": "Answer questions about the provided image. Base every answer only on what can be seen. Respond with interleaved text and audio."},
    {"role": "user", "content": [{"type": "image", "image": "bee.jpg"}, {"type": "text", "text": "Is this insect useful for the garden? Why?"}]},
])
print("💬", answer)
play(speech)

# 🎬 Video

Video is sampled at 2 frames per second; consecutive frames are paired and each pair is prefixed with its timestamp.
System prompt: `Answer questions about the video using what you see and hear.`

To include the **soundtrack**, pass `load_audio_from_video=True` and `use_audio_in_video=True`: the processor reads the
video's audio track and interleaves it with the frames, so each pair of frames is followed by the sound that goes with it.

In [ ]:
from huggingface_hub import hf_hub_download

video = hf_hub_download("raushan-testing-hf/videos-test", "sample_demo_1.mp4", repo_type="dataset")


@torch.inference_mode()
def ask_video(question, use_audio=False):
    extra = dict(load_audio_from_video=True, processor_kwargs={"use_audio_in_video": True}) if use_audio else {}
    inputs = processor.apply_chat_template(
        [
            {"role": "system", "content": "Answer questions about the video using what you see and hear."},
            {"role": "user", "content": [{"type": "video", "video": video}, {"type": "text", "text": question}]},
        ],
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt", **extra,
    ).to(device)
    out = model.generate(**inputs, max_new_tokens=256, generation_mode="text")
    return processor.decode(out.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print("🎬 Frames only:\n", ask_video("What happens in this video?"))
print("\n🎬🔊 Frames and soundtrack:\n", ask_video("What happens in this video, and what can you hear?", use_audio=True))

# 🎧 Audio understanding

Open-ended questions about sounds, music and speech need no system prompt: put the audio and the question in the user turn.
For multiple choice, use the system prompt `Answer questions about the provided audio clip. Base every answer only on what can be heard.`
and end the user turn with `Please choose the answer from the options.`

In [ ]:
answer, _ = generate([
    {"role": "user", "content": [
        {"type": "audio", "audio": "german.wav"},
        {"type": "text", "text": "What language is spoken, is the speaker a man or a woman, and what is the message about?"},
    ]},
])
print("💡", answer)

In [ ]:
answer, _ = generate([
    {"role": "system", "content": "Answer questions about the provided audio clip. Base every answer only on what can be heard."},
    {"role": "user", "content": [
        {"type": "audio", "audio": "german.wav"},
        {"type": "text", "text": "What activity does the speaker suggest?\nA. Cooking a new recipe\nB. Drawing animals with closed eyes\nC. Planting flowers\nD. Learning a dance\nPlease choose the answer from the options."},
    ]},
])
print("💡", answer)

# 🔧 Tool calling: book a table from a restaurant photo

Show the model a restaurant photo and ask by voice to book a table. The model reads the restaurant name from the
sign and combines it with the city, date, time, party size, and reservation name in the spoken request.

Pass the `book_table` schema to `apply_chat_template()` with `tools=...`. The model returns the proposed call
between `<tool_call>` and `</tool_call>`. This example only displays the call; it does not make a reservation.

Photo: [Café de Flore, Paris](https://commons.wikimedia.org/wiki/File:Cafe_de_Flore_Paris_France.jpg) by ayustety,
[CC BY-SA 2.0](https://creativecommons.org/licenses/by-sa/2.0/), resized for display.

In [ ]:
from datetime import date, timedelta
from urllib.request import Request, urlopen
from PIL import Image

tools = [{
    "name": "book_table",
    "description": "Book a table at a restaurant using its name and city.",
    "parameters": {
        "type": "object",
        "properties": {
            "restaurant_name": {"type": "string"},
            "city": {"type": "string"},
            "date": {"type": "string", "description": "Local date, YYYY-MM-DD."},
            "time": {"type": "string", "description": "Local time, HH:MM in 24-hour format."},
            "party_size": {"type": "integer", "minimum": 1},
            "reservation_name": {"type": "string"},
        },
        "required": ["restaurant_name", "city", "date", "time", "party_size", "reservation_name"],
    },
}]

photo_url = "https://upload.wikimedia.org/wikipedia/commons/8/80/Cafe_de_Flore_Paris_France.jpg"
with urlopen(Request(photo_url, headers={"User-Agent": "LFM3Cookbook"}), timeout=30) as response:
    restaurant_image = Image.open(response).convert("RGB")
restaurant_image.save("restaurant.jpg")
display(restaurant_image.resize((900, 600)))

booking_date = date.today() + timedelta(days=7)
_, spoken_request = generate([
    {"role": "system", "content": "Read this with a male voice in British English."},
    {"role": "user", "content": f"Please book a table for two at this restaurant in Paris on November two, at seven in the evening, under Alex Taylor."},
])
sf.write("request.wav", spoken_request, SR)
play(spoken_request)

inputs = processor.apply_chat_template(
    [
        {"role": "system", "content": "Respond with function calls. Use the tools to help the user complete their goal.\n\n Current date: 2026-10-27"},
        {"role": "user", "content": [
            {"type": "image", "image": "restaurant.jpg"},
            {"type": "audio", "audio": "request.wav"},
        ]},
    ],
    tools=tools, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
).to(device)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=512, generation_mode="text")
booking_call = processor.decode(out.sequences[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("🔧 Proposed booking call (not executed):", booking_call)

# 🎛️ Controlling the output modality

The same question, answered in text and in interleaved text and speech. `generation_mode` overrides whatever the model would choose on its own.
Use `generation_mode="audio"` (speech only) for reading text aloud; for open-ended answers, prefer `"interleaved"`.

In [ ]:
conversation = [
    {"role": "user", "content": [{"type": "image", "image": "bee.jpg"}, {"type": "text", "text": "Describe this picture in one sentence."}]},
]
for mode in ["text", "interleaved"]:
    torch.manual_seed(0)
    reply, speech = generate(conversation, generation_mode=mode)
    print(f"🎛️ {mode:12s} text: {reply!r}")
    if speech is not None:
        play(speech)

# 🚀 Next steps

- Fine-tune LFM3-Omni on your own data: see [`lfm3_omni_vqa_sft_with_transformers.ipynb`](../finetuning/notebooks/lfm3_omni_vqa_sft_with_transformers.ipynb).
- Speech-only workloads: [`lfm3_audio_capabilities.ipynb`](lfm3_audio_capabilities.ipynb) covers LFM3-Audio.
- Documentation: [docs.liquid.ai](https://docs.liquid.ai/lfm) · Community: [Discord](https://discord.com/invite/liquid-ai)